Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: CCS4310 – Deep Learning  
Stage: Fashion Generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

# Fashion Generation - Stable Diffusion + PEFT LoRA

All expensive operations are explicitly gated and disabled by default. Raw data are never modified.

Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: CCS4310 – Deep Learning  
Stage: Fashion Generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

## Configuration and reproducibility

In [ ]:
from pathlib import Path
import os, json, random
import numpy as np, pandas as pd
import torch
SEED=42; random_state=SEED; FAST_DEV_RUN=True; FAST_DEV_TRAIN_SAMPLES=100; FAST_DEV_VALIDATION_SAMPLES=20
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
PROJECT_ROOT=Path.cwd()
for candidate in [PROJECT_ROOT,*PROJECT_ROOT.parents]:
    if (candidate/"data"/"raw"/"deepfashion").exists(): PROJECT_ROOT=candidate; break
BASE_MODEL=os.environ.get("DEEPFASHION_BASE_MODEL","runwayml/stable-diffusion-v1-5")
IMAGE_SIZE=512; TRAIN_BATCH_SIZE=1; GRADIENT_ACCUMULATION_STEPS=4; LEARNING_RATE=1e-4; NUM_EPOCHS=1
LORA_RANK=8; LORA_ALPHA=16; LORA_DROPOUT=.05; FID_MIN_IMAGES=50
MIXED_PRECISION=os.environ.get("MIXED_PRECISION","no"); DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_CSV_PATH=PROJECT_ROOT/"data/interim/deepfashion_generation_train.csv"; VALIDATION_CSV_PATH=PROJECT_ROOT/"data/interim/deepfashion_generation_validation.csv"; NUM_WORKERS=int(os.environ.get("DEEPFASHION_NUM_WORKERS", "0"))
IMAGE_ROOT=PROJECT_ROOT/"data/raw/deepfashion/images"; CHECKPOINT_DIR=PROJECT_ROOT/"models/generation/deepfashion_lora_checkpoints"; RESUME_CHECKPOINT=None
BASELINE_DIR=PROJECT_ROOT/"outputs/generated_designs/stable_diffusion_baseline"; LORA_DIR=PROJECT_ROOT/"outputs/generated_designs/stable_diffusion_lora"; METRICS_DIR=PROJECT_ROOT/"outputs/metrics"; MODELS_DIR=PROJECT_ROOT/"models/generation"; FINAL_LORA_DIR=MODELS_DIR/"deepfashion_lora_final"; TRAINING_CONFIG_PATH=MODELS_DIR/"training_config.json"
RUN_BASELINE_INFERENCE=False; RUN_TRAINING=False; RUN_LORA_INFERENCE=False; RUN_METRIC_EVALUATION=False
config={"base_model":BASE_MODEL,"seed":SEED,"image_size":IMAGE_SIZE,"batch_size":TRAIN_BATCH_SIZE,"gradient_accumulation_steps":GRADIENT_ACCUMULATION_STEPS,"learning_rate":LEARNING_RATE,"num_epochs":NUM_EPOCHS,"lora_rank":LORA_RANK,"lora_alpha":LORA_ALPHA,"lora_dropout":LORA_DROPOUT,"mixed_precision":MIXED_PRECISION,"fast_dev_run":FAST_DEV_RUN,"fast_dev_train_samples":FAST_DEV_TRAIN_SAMPLES,"fast_dev_validation_samples":FAST_DEV_VALIDATION_SAMPLES,"device":str(DEVICE),"resume_checkpoint":RESUME_CHECKPOINT,"flags":{"baseline":RUN_BASELINE_INFERENCE,"training":RUN_TRAINING,"lora":RUN_LORA_INFERENCE,"metrics":RUN_METRIC_EVALUATION}}
print("device:",DEVICE,"flags:",config["flags"])


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: CCS4310 – Deep Learning  
Stage: Fashion Generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

## Lazy Dataset, aspect-ratio-safe transforms, and DataLoader

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
class DeepFashionLazyDataset(Dataset):
    def __init__(self, frame, image_root, size=IMAGE_SIZE):
        self.frame=frame.reset_index(drop=True); self.image_root=Path(image_root)
        self.transform=transforms.Compose([transforms.Resize(size, interpolation=transforms.InterpolationMode.BICUBIC), transforms.CenterCrop((size,size)), transforms.ToTensor(), transforms.Normalize([.5]*3,[.5]*3)])
    def __len__(self): return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        with Image.open(self.image_root/str(row.image_path)) as im: pixels=self.transform(im.convert("RGB"))
        return {"pixel_values":pixels,"caption":str(row.caption),"image_path":str(row.image_path)}
assert TRAIN_CSV_PATH.exists(), f"Missing training manifest: {TRAIN_CSV_PATH}. Run Notebook 2 first."
assert VALIDATION_CSV_PATH.exists(), f"Missing validation manifest: {VALIDATION_CSV_PATH}. Run Notebook 2 first."
train_df=pd.read_csv(TRAIN_CSV_PATH); val_df=pd.read_csv(VALIDATION_CSV_PATH)
required_columns={"image_path","caption","split"}; assert required_columns.issubset(train_df.columns) and required_columns.issubset(val_df.columns)
assert train_df["caption"].fillna("").str.strip().ne("").all() and val_df["caption"].fillna("").str.strip().ne("").all()
assert set(train_df["image_path"]).isdisjoint(set(val_df["image_path"]))
val_df=val_df.head(FAST_DEV_VALIDATION_SAMPLES) if FAST_DEV_RUN else val_df
if FAST_DEV_RUN: train_df=train_df[train_df.get("split", "train").eq("train")].head(FAST_DEV_TRAIN_SAMPLES) if "split" in train_df else train_df.head(FAST_DEV_TRAIN_SAMPLES)
dataset=DeepFashionLazyDataset(train_df,IMAGE_ROOT) if len(train_df) else None
loader=DataLoader(dataset,batch_size=TRAIN_BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda") if dataset else None


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: CCS4310 – Deep Learning  
Stage: Fashion Generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

## Baseline inference (same deterministic prompts/seeds)

In [ ]:
prompts=list(val_df.caption.head(4)) if len(val_df) else ["a contemporary fashion garment, studio product photograph"]
seeds=[SEED+i for i in range(len(prompts))]; baseline_results=[]
if RUN_BASELINE_INFERENCE:
    from diffusers import StableDiffusionPipeline
    BASELINE_DIR.mkdir(parents=True,exist_ok=True)
    pipe=StableDiffusionPipeline.from_pretrained(BASE_MODEL,torch_dtype=torch.float16 if DEVICE.type=="cuda" else torch.float32).to(DEVICE)
    for prompt,seed in zip(prompts,seeds):
        generator=torch.Generator(device=DEVICE).manual_seed(seed)
        out=BASELINE_DIR/f"seed_{seed}.png"; pipe(prompt,generator=generator).images[0].save(out)
        baseline_results.append({"prompt":prompt,"seed":seed,"baseline_image":str(out)})


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: Fashion generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

## Inspect attention modules and attach PEFT LoRA

In [ ]:
def attention_modules(unet):
    return [(name,module.__class__.__name__) for name,module in unet.named_modules() if "attn" in name.lower() or "attention" in name.lower()]
attention_report=[]; targets=[]
if RUN_TRAINING or RUN_LORA_INFERENCE:
    from diffusers import UNet2DConditionModel
    inspect_unet=UNet2DConditionModel.from_pretrained(BASE_MODEL,subfolder="unet")
    attention_report=attention_modules(inspect_unet)
    discovered_targets=set()
    for name,_ in attention_report:
        for suffix in ("to_q","to_k","to_v","to_out.0"):
            if name.endswith(suffix): discovered_targets.add(suffix)
    targets=sorted(discovered_targets)
    if not targets: raise RuntimeError("No injectable UNet attention projections were discovered")
    print("attention modules:",attention_report[:20],"LoRA targets:",targets)


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: Fashion generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

## Gated Diffusers/PEFT LoRA training with AMP, tqdm, checkpointing, and resume

In [ ]:
loss_history=[]
if RUN_TRAINING:
    from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
    from transformers import CLIPTextModel, CLIPTokenizer
    from peft import LoraConfig, get_peft_model
    from tqdm.auto import tqdm
    tokenizer=CLIPTokenizer.from_pretrained(BASE_MODEL,subfolder="tokenizer")
    text_encoder=CLIPTextModel.from_pretrained(BASE_MODEL,subfolder="text_encoder").to(DEVICE)
    vae=AutoencoderKL.from_pretrained(BASE_MODEL,subfolder="vae").to(DEVICE)
    unet=inspect_unet.to(DEVICE)
    vae.requires_grad_(False); text_encoder.requires_grad_(False); unet.requires_grad_(False)
    lora_config=LoraConfig(r=LORA_RANK,lora_alpha=LORA_ALPHA,lora_dropout=LORA_DROPOUT,target_modules=targets,bias="none")
    if RESUME_CHECKPOINT and not Path(RESUME_CHECKPOINT).exists(): raise FileNotFoundError(f"Requested RESUME_CHECKPOINT does not exist: {RESUME_CHECKPOINT}")
    if RESUME_CHECKPOINT:
        from peft import PeftModel
        unet=PeftModel.from_pretrained(unet,str(RESUME_CHECKPOINT),is_trainable=True,adapter_name="fashion_lora")
    else: unet=get_peft_model(unet,lora_config,adapter_name="fashion_lora")
    unet.print_trainable_parameters(); trainable=sum(p.numel() for p in unet.parameters() if p.requires_grad); total=sum(p.numel() for p in unet.parameters())
    scheduler=DDPMScheduler.from_pretrained(BASE_MODEL,subfolder="scheduler"); CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True)
    start_epoch=0; resume_state=None
    if RESUME_CHECKPOINT:
        resume_state=torch.load(Path(RESUME_CHECKPOINT)/"optimizer.pt",map_location=DEVICE); start_epoch=int(resume_state.get("epoch",0))
    optimizer=torch.optim.AdamW((p for p in unet.parameters() if p.requires_grad),lr=LEARNING_RATE)
    if resume_state is not None: optimizer.load_state_dict(resume_state["optimizer"])
    use_amp=DEVICE.type=="cuda" and MIXED_PRECISION in {"fp16","bf16"}; amp_dtype=torch.float16 if MIXED_PRECISION=="fp16" else torch.bfloat16; scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    for epoch in range(start_epoch,NUM_EPOCHS):
        optimizer.zero_grad(set_to_none=True)
        for step,batch in enumerate(tqdm(loader,desc=f"epoch {epoch+1}/{NUM_EPOCHS}")):
            pixels=batch["pixel_values"].to(DEVICE); ids=tokenizer(list(batch["caption"]),padding="max_length",max_length=tokenizer.model_max_length,truncation=True,return_tensors="pt").input_ids.to(DEVICE)
            with torch.no_grad(): latents=vae.encode(pixels).latent_dist.sample()*vae.config.scaling_factor; hidden=text_encoder(ids)[0]
            noise=torch.randn_like(latents); timesteps=torch.randint(0,scheduler.config.num_train_timesteps,(latents.shape[0],),device=DEVICE).long(); noisy=scheduler.add_noise(latents,noise,timesteps)
            with torch.autocast(device_type="cuda",dtype=amp_dtype,enabled=use_amp): loss=torch.nn.functional.mse_loss(unet(noisy,timesteps,encoder_hidden_states=hidden).sample,noise)/GRADIENT_ACCUMULATION_STEPS
            scaler.scale(loss).backward()
            if (step+1)%GRADIENT_ACCUMULATION_STEPS==0 or (step+1)==len(loader): scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            loss_history.append(float(loss.detach().cpu()*GRADIENT_ACCUMULATION_STEPS)); tqdm.write(f"step={step} loss={loss_history[-1]:.5f}")
        ckpt=CHECKPOINT_DIR/f"epoch_{epoch+1}"; unet.save_pretrained(ckpt); torch.save({"epoch":epoch+1,"optimizer":optimizer.state_dict()},ckpt/"optimizer.pt")
        print(f"LoRA parameters: {trainable:,} trainable / {total:,} total UNet")
    FINAL_LORA_DIR.mkdir(parents=True,exist_ok=True); unet.save_pretrained(FINAL_LORA_DIR)
    config.update({"trainable_parameters":trainable,"total_unet_parameters":total,"attention_targets":targets,"loss_history":loss_history,"final_lora_dir":str(FINAL_LORA_DIR)}); METRICS_DIR.mkdir(parents=True,exist_ok=True); TRAINING_CONFIG_PATH.write_text(json.dumps(config,indent=2),encoding="utf-8")


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: Fashion generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

## LoRA inference and side-by-side visualization

In [ ]:
comparison=pd.DataFrame(columns=["prompt","seed","baseline_image","lora_image"])
if RUN_LORA_INFERENCE and not FINAL_LORA_DIR.exists():
    print(f"LoRA adapter not found at {FINAL_LORA_DIR}; skipping LoRA inference.")
elif RUN_LORA_INFERENCE:
    from diffusers import StableDiffusionPipeline
    LORA_DIR.mkdir(parents=True,exist_ok=True); rows=[]
    pipe=StableDiffusionPipeline.from_pretrained(BASE_MODEL,torch_dtype=torch.float16 if DEVICE.type=="cuda" else torch.float32).to(DEVICE)
    from peft import LoraConfig, PeftModel
    pipe.unet.requires_grad_(False)
    try:
        pipe.unet=PeftModel.from_pretrained(pipe.unet,str(FINAL_LORA_DIR),is_trainable=False,adapter_name="fashion_lora")
    except (ImportError,TypeError,ValueError,KeyError) as exc:
        raise RuntimeError(f"Saved PEFT LoRA adapter is incompatible with the installed PEFT API: {exc}") from exc
    if not hasattr(pipe.unet,"set_adapter"): raise RuntimeError("Loaded PEFT model does not expose set_adapter")
    pipe.unet.set_adapter("fashion_lora")
    for prompt,seed in zip(prompts,seeds):
        out=LORA_DIR/f"seed_{seed}.png"; pipe(prompt,generator=torch.Generator(device=DEVICE).manual_seed(seed)).images[0].save(out)
        base=BASELINE_DIR/f"seed_{seed}.png"; rows.append({"prompt":prompt,"seed":seed,"baseline_image":str(base) if base.exists() else "","lora_image":str(out)})
    comparison=pd.DataFrame(rows)
valid_comparison=comparison[comparison.baseline_image.map(lambda x:bool(x) and Path(x).exists()) & comparison.lora_image.map(lambda x:bool(x) and Path(x).exists())]
if not valid_comparison.empty:
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(len(valid_comparison),2,figsize=(8,4*len(valid_comparison))); axes=np.atleast_2d(axes)
    for i,(_,row) in enumerate(valid_comparison.iterrows()):
        for j,key in enumerate(("baseline_image","lora_image")):
            with Image.open(row[key]) as image: axes[i,j].imshow(image.convert("RGB"))
            axes[i,j].set_title(key); axes[i,j].axis("off")
    fig.tight_layout(); fig.savefig(LORA_DIR/"baseline_vs_lora.png",dpi=150)


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: Fashion generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

## Optional real CLIPScore, LPIPS, and FID

In [ ]:
def evaluate_optional_metrics(reference_images, generated_images, captions):
    result={"status":"unavailable"}
    paired=[(Path(ref),Path(gen),caption) for ref,gen,caption in zip(reference_images,generated_images,captions) if Path(ref).exists() and Path(gen).exists()]
    reference_images=[ref for ref,_,_ in paired]; generated_images=[gen for _,gen,_ in paired]; captions=[caption for _,_,caption in paired]
    if not generated_images: result["status"]="skipped: no valid paired images"; return result
    if len(generated_images) < FID_MIN_IMAGES or len(reference_images) < FID_MIN_IMAGES: result["fid_status"]=f"skipped: FID requires at least {FID_MIN_IMAGES} valid images in each set"; print(result["fid_status"])
    try:
        from torchmetrics.multimodal.clip_score import CLIPScore
        metric=CLIPScore(model_name_or_path="openai/clip-vit-base-patch32").to(DEVICE)
        for path,caption in zip(generated_images,captions):
            arr=np.asarray(Image.open(path).convert("RGB")); metric.update(torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(DEVICE),[caption])
        result["clipscore"]=float(metric.compute().cpu())
    except Exception as exc: result["clipscore_error"]=str(exc)
    try:
        from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
        lpips=LearnedPerceptualImagePatchSimilarity(net_type="vgg").to(DEVICE); pairs=min(len(reference_images),len(generated_images))
        if pairs == 0: result["lpips_status"]="skipped: no valid paired images"; return result
        for ref,gen in zip(reference_images[:pairs],generated_images[:pairs]):
            with Image.open(ref) as ref_image, Image.open(gen) as gen_image:
                target_size=(min(ref_image.width,gen_image.width),min(ref_image.height,gen_image.height)); ref_array=np.asarray(ref_image.convert("RGB").resize(target_size)); gen_array=np.asarray(gen_image.convert("RGB").resize(target_size))
            a=torch.from_numpy(ref_array).permute(2,0,1).float().div(127.5).sub(1).unsqueeze(0).to(DEVICE); b=torch.from_numpy(gen_array).permute(2,0,1).float().div(127.5).sub(1).unsqueeze(0).to(DEVICE); lpips.update(a,b)
        result["lpips"]=float(lpips.compute().cpu())
    except Exception as exc: result["lpips_error"]=str(exc)
    try:
        from torchmetrics.image.fid import FrechetInceptionDistance
        if len(generated_images) >= FID_MIN_IMAGES and len(reference_images) >= FID_MIN_IMAGES:
            fid=FrechetInceptionDistance(feature=2048).to(DEVICE)
            for paths,real in ((reference_images,True),(generated_images,False)):
                for path in paths: fid.update(torch.from_numpy(np.asarray(Image.open(path).convert("RGB"))).permute(2,0,1).unsqueeze(0).to(DEVICE),real=real)
            result["fid"]=float(fid.compute().cpu())
    except Exception as exc: result["fid_error"]=str(exc)
    result["status"]="computed where optional dependencies succeeded; unavailable metrics are recorded honestly"; return result
metrics={}
if RUN_METRIC_EVALUATION and not comparison.empty:
    metrics=evaluate_optional_metrics(val_df.image_path.head(len(comparison)).map(lambda x:str(IMAGE_ROOT/x)),comparison.lora_image.tolist(),comparison.prompt.tolist())


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: Fashion generation  
Purpose: Gated Stable Diffusion generation, LoRA training, and evaluation.

## Human review template and real results only

In [ ]:
HUMAN_REVIEW_OUTPUT=METRICS_DIR/"fashion_generation_human_review.csv"; RESULTS_CSV=METRICS_DIR/"fashion_generation_results.csv"; METRICS_DIR.mkdir(parents=True,exist_ok=True)
review_columns=["sample_id","prompt","baseline_image","lora_image","fashion_relevance","prompt_alignment","visual_quality","overall_preference","comments"]
if not HUMAN_REVIEW_OUTPUT.exists(): pd.DataFrame(columns=review_columns).to_csv(HUMAN_REVIEW_OUTPUT,index=False)
real_metrics={k:v for k,v in metrics.items() if isinstance(v,(int,float)) and not isinstance(v,bool)}
result_rows=[{**row,**metrics} for row in comparison.to_dict("records")] if not comparison.empty else [{"status":metrics.get("status","skipped: metric evaluation disabled"),**metrics}]
pd.DataFrame(result_rows).to_csv(RESULTS_CSV,index=False)
TRAINING_CONFIG_PATH.parent.mkdir(parents=True,exist_ok=True); TRAINING_CONFIG_PATH.write_text(json.dumps(config,indent=2),encoding="utf-8")
print("review template:",HUMAN_REVIEW_OUTPUT)
